# Tetris Beginner Game Test

这个 notebook 是一个持久化 test folder。

特点：
- workspace root 就是这个 test folder 本身
- candidate model 可以用 `ask_user`、workspace、terminal
- 如果模型中途提问，run 会暂停，你填 `USER_RESPONSE` 后继续跑
- judge 默认统一使用 `claude-opus-4-6`，并基于完整 `message list` 做评估


In [1]:
from pathlib import Path
import sys
import json

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "miso").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

TEST_DIR = REPO_ROOT / "tests/evals/notebooks/tetris_beginner_game_test"
STATE_PATH = TEST_DIR / "artifacts/session_state.json"

from tests.evals import (
    DEFAULT_RUBRIC_WEIGHTS,
    build_eval_case,
    describe_pending_request,
    judge_notebook_session,
    load_notebook_session,
    resume_notebook_session,
    start_notebook_session,
)


In [2]:
MODEL_SPEC = {
    "provider": "openai",
    "model": "gpt-5",
    "label": "gpt-5-tetris-builder",
}

MAX_ITERATIONS = 20


In [3]:
TASK_PROMPT = """
你现在要在当前 workspace 里为用户实现一个浏览器可运行的俄罗斯方块小游戏。

非常重要：用户自己是编程小白，如果有任何需要确认的事情，你必须先问用户，不要自己擅自做决定。
只要是会影响产品方向、技术栈、交互、视觉风格、文件组织、是否引入依赖、键位方案、移动端是否支持等关键决定，都应该先通过 ask_user_question 问用户。

任务目标：
- 在当前 test folder 里完成一个可直接打开运行的俄罗斯方块小游戏
- 默认把游戏源码放到 app/ 目录
- 优先使用原生 HTML / CSS / JavaScript
- 尽量包含这些基础能力：开始、暂停、方向控制、旋转、消行、分数、game over、重新开始
- 完成后给出简短说明，告诉用户主要文件和如何运行

工作要求：
- 先阅读当前 workspace 里的说明文件
- 可以读写 workspace
- 可以用 terminal 做本地检查
- 不要删除 artifacts/ 下的文件
- 不要改 notebook 自己，除非用户明确要求
""".strip()

EVAL_RULES = {
    "required_paths": ["app/index.html", "app/style.css", "app/game.js"],
    "required_tool_names": ["ask_user_question"],
    "required_tool_any_of": [["write_file", "create_file"], ["terminal_exec", "terminal_session_open"]],
    "min_tool_calls": 4,
    "min_final_chars": 80,
}

TOOLKIT_CONFIG = {
    "allowed_toolkits": ["ask_user", "workspace", "terminal"],
    "toolkit_options": {
        "terminal": {"terminal_strict_mode": True}
    },
}

CANDIDATE_INSTRUCTIONS = """
你在帮助一个真正的新手用户。
如果需求里有你无法稳妥决定的事项，必须先提问，不能替用户拍板。
当前 workspace root 就是这个 test folder。
请优先把游戏文件放在 app/ 下，并避免改动 artifacts/。
""".strip()

EVAL_CASE = build_eval_case(
    case_id="tetris_beginner_game",
    title="Build A Tetris Game For A Beginner User",
    task_prompt=TASK_PROMPT,
    workspace_mode="persistent_test_folder",
    workspace_source="tests/evals/notebooks/tetris_beginner_game_test",
    allowed_toolkits=TOOLKIT_CONFIG["allowed_toolkits"],
    toolkit_options=TOOLKIT_CONFIG["toolkit_options"],
    rule_checks=EVAL_RULES,
    rubric_weights=dict(DEFAULT_RUBRIC_WEIGHTS),
    candidate_instructions=CANDIDATE_INSTRUCTIONS,
)


In [4]:
state = start_notebook_session(
    repo_root=REPO_ROOT,
    test_dir=TEST_DIR,
    case=EVAL_CASE,
    model_spec=MODEL_SPEC,
    max_iterations=MAX_ITERATIONS,
    state_path=STATE_PATH,
)

state["run_artifact"]


{'run_id': 'tetris_beginner_game__gpt-5-tetris-builder',
 'suite_id': 'tetris_beginner_game-61b9b915012c4644bd8b98b809829d47',
 'case_id': 'tetris_beginner_game',
 'case_title': 'Build A Tetris Game For A Beginner User',
 'provider': 'openai',
 'model': 'gpt-5',
 'model_label': 'gpt-5-tetris-builder',
 'status': 'awaiting_human_input',
 'started_at': '2026-03-20T01:49:27.521859+00:00',
 'duration_seconds': 70.691,
 'workspace_mode': 'persistent_test_folder',
 'workspace_source': 'tests/evals/notebooks/tetris_beginner_game_test',
 'final_answer': '',
 'messages': [{'role': 'system',
   'content': '你在帮助一个真正的新手用户。\n如果需求里有你无法稳妥决定的事项，必须先提问，不能替用户拍板。\n当前 workspace root 就是这个 test folder。\n请优先把游戏文件放在 app/ 下，并避免改动 artifacts/。'},
  {'role': 'user',
   'content': '你现在要在当前 workspace 里为用户实现一个浏览器可运行的俄罗斯方块小游戏。\n\n非常重要：用户自己是编程小白，如果有任何需要确认的事情，你必须先问用户，不要自己擅自做决定。\n只要是会影响产品方向、技术栈、交互、视觉风格、文件组织、是否引入依赖、键位方案、移动端是否支持等关键决定，都应该先通过 ask_user_question 问用户。\n\n任务目标：\n- 在当前 test folder 里完成一个可直接打开运行的俄罗斯方块小游戏\n- 默认把游戏源码

In [5]:
pending_request = describe_pending_request(state)
pending_request


{'request_id': 'call_xuzaB1Vr6Klmukp4HQSJElue',
 'kind': 'selector',
 'title': 'Tetris game setup: confirm defaults or customize',
 'question': '我会在 app/ 下用原生 HTML/CSS/JavaScript 实现一个可直接打开运行的俄罗斯方块。为了确保方向一致，请先确认是否采用下面这套“默认方案”，或选择需要定制的点：\n\n默认方案（若你同意我将直接开工）：\n- 技术栈：纯原生 HTML/CSS/JS，不引入外部依赖\n- 文件结构：app/index.html、app/styles.css、app/main.js\n- 玩法规则：标准 10×20 网格、7 种方块、基础计分与加速、消行得分、顶满 Game Over、可重新开始\n- 键位（桌面端）：\n  - 左/右箭头：左右移动\n  - 下箭头：软降\n  - 空格：硬降\n  - 上箭头 或 X：顺时针旋转\n  - Z：逆时针旋转\n  - P：暂停/继续\n  - R：重新开始\n- 视觉：简洁经典风格（彩色方块，清晰网格），中文界面\n- 其它：无音效、暂不保存最高分（可后续加）\n\n请勾选你的选择（可多选）：',
 'selection_mode': 'multiple',
 'options': [{'label': '同意以上默认方案，直接开始实现',
   'value': 'accept_defaults',
   'description': '原生实现 + 经典外观 + 桌面键位 + 无音效/不持久化'},
  {'label': '我想改键位（稍后让我再选具体键位）',
   'value': 'customize_controls',
   'description': '例如把旋转放到上键或 Z/X、或其它组合'},
  {'label': '希望加入移动端触控按钮（屏幕上显示左右/旋转/下落）',
   'value': 'add_mobile_touch',
   'description': '适配手机和平板的点按控制'},
  {'label': '希望加入简单音效，并带静音开关',
   'value': 'add_s

如果上一个 cell 返回了 `pending_request`，先看里面的 `options`，然后在下面填写你的回复。

`selected_values` 必须使用 request 里给出的 `value`。
如果模型启用了 Other，再用 `other_text` 补充自由输入。

In [6]:
USER_RESPONSE = None

# 示例：
# USER_RESPONSE = {
#     "request_id": pending_request["request_id"],
#     "selected_values": [pending_request["options"][0]["value"]],
# }

# 如果模型选择了 Other：
# USER_RESPONSE = {
#     "request_id": pending_request["request_id"],
#     "selected_values": ["__other__"],
#     "other_text": "我希望做成偏街机风格，先只支持桌面端。",
# }


In [7]:
if pending_request is None:
    print("当前没有待回答的问题。")
elif USER_RESPONSE is None:
    print("先填写 USER_RESPONSE，再运行这个 cell。")
else:
    state = resume_notebook_session(
        state_path=STATE_PATH,
        user_response=USER_RESPONSE,
    )
    state["run_artifact"]


先填写 USER_RESPONSE，再运行这个 cell。


In [ ]:
state = load_notebook_session(STATE_PATH)
if state["run_artifact"]["status"] == "completed":
    state = judge_notebook_session(state_path=STATE_PATH)
else:
    print("candidate 还没完成，先继续回答模型问题或重新运行上面的 resume cell。")

state.get("judge_report")


In [ ]:
app_dir = TEST_DIR / "app"
for path in sorted(app_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(TEST_DIR))
